# Module 1 Instructor Colab Notebook: Vectors, Inner Products, and Norms

## A Mathematical Bridge to Artificial Intelligence

This module develops the mathematical language used to represent data and compare embeddings in artificial intelligence. It is designed as a lecture-ready instructor notebook: definitions are motivated before they are formalized, calculations are shown step by step, code verifies rather than replaces the mathematics, and direct AI applications appear throughout.

**Recommended pacing:** two or three class meetings. Cells labeled **Core** form the essential pathway. Cells labeled **Extension** may be used for a longer class, enrichment, or review.


> **How to use this notebook in class**
>
> Run the setup cells before class. During class, pause at each prediction prompt before running the next code cell. Use the interactive cells selectively. Instructor solution cells open answers with a button so students can work before the solution is displayed.

### Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Scalars, vectors, matrices, and tensors | Numerical data representations and embeddings |
| 2 | Dot products, angles, orthogonality, and projection | Linear neurons and feature matching |
| 3 | L1 norm, L2 norm, distance, and regularization | Comparing data and controlling model weights |
| 4 | Cosine similarity | Embedding search and retrieval-augmented generation |
| Integrated lab | Combine all four sections | A small, transparent retrieval system |


## Module learning objectives

By the end of Module 1, students should be able to:

1. distinguish a scalar, vector, matrix, and tensor and state the shape of each object;
2. explain why AI data are converted into numerical arrays;
3. compute a dot product entry by entry and interpret it algebraically and geometrically;
4. determine whether two vectors are orthogonal;
5. compute vector projections and verify that the residual is orthogonal to the projection direction;
6. compute and compare L1 and L2 norms;
7. compute L1 and L2 distances between vectors;
8. explain the role of L1 and L2 penalties in regularization;
9. compute cosine similarity by hand and use it to rank embeddings; and
10. identify common failure cases, including incompatible dimensions and the zero-vector problem.


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable helpers used throughout the notebook.
def reveal_markdown(button_text, solution_markdown):
    # Create a one-button instructor reveal, with a plain fallback.
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def cosine_similarity(u, v):
    # Return cosine similarity; return NaN when either vector is zero.
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    denominator = np.linalg.norm(u, 2) * np.linalg.norm(v, 2)
    if np.isclose(denominator, 0.0):
        return np.nan
    return float(np.dot(u, v) / denominator)


def plot_two_vectors(u, v, labels=("u", "v"), title="Two vectors"):
    # Plot two two-dimensional vectors from the origin.
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    limit = max(1.0, np.max(np.abs(np.concatenate([u, v])))) + 1.0
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.quiver(0, 0, u[0], u[1], angles="xy", scale_units="xy", scale=1,
              color="#1f77b4", width=0.012, label=labels[0])
    ax.quiver(0, 0, v[0], v[1], angles="xy", scale_units="xy", scale=1,
              color="#d62728", width=0.012, label=labels[1])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("first coordinate")
    ax.set_ylabel("second coordinate")
    ax.set_title(title)
    ax.legend()
    plt.show()


> **Notation and code convention**
>
> Mathematical vectors are treated as column vectors when formulas are written. In NumPy, a one-dimensional array has shape `(n,)`; it has entries but no explicit row-or-column orientation. When orientation matters, we reshape it to `(n, 1)` or `(1, n)`.

---

# Section 1: Scalars, Vectors, Matrices, and Tensors

## Learning objectives

Students should be able to distinguish the four data structures, identify their order and shape, interpret indices, and connect each structure to a type of AI data.


## 1. Why AI begins with numerical representations

A computer cannot directly calculate with the meaning of a sentence, the mood of a song, or the visual content of a photograph. The data must first be encoded as numbers. Once data are numerical, mathematical operations can compare, transform, and learn from those numbers.

This produces a hierarchy:

1. one number is a **scalar**;
2. an ordered list of numbers is a **vector**;
3. a rectangular table of numbers is a **matrix**; and
4. a multi-index numerical array is commonly called a **tensor** in machine learning.

> **Important distinction**
>
> In machine learning software, *tensor* often means a multidimensional numerical array. In advanced mathematics, the word has a more formal coordinate-independent meaning. This bridge course uses the common machine-learning convention.


### What is an embedding?

An **embedding** is a numerical representation of an object, such as a word, sentence, image, song, or document. An embedding model converts the original object into a vector:

$$
\text{object}
\longrightarrow
\text{embedding vector}.
$$

For example, a sentence might be represented by

$$
\mathbf{e}
=
\begin{bmatrix}
0.72\\
-0.15\\
0.48\\
0.31
\end{bmatrix}
\in\mathbb{R}^{4}.
$$

The coordinates are numerical features learned by the embedding model. In most modern AI systems, an individual coordinate does not have a simple interpretation such as “topic,” “sentiment,” or “importance.” Instead, information is distributed across many coordinates.

The goal is generally to place related objects in similar locations or directions in the embedding space. For example, two sentences discussing similar mathematical ideas should usually have more similar embeddings than a mathematics sentence and an unrelated hiking sentence. Their similarity can be measured using dot products, vector distances, or cosine similarity.

If `m` objects are each represented by an embedding with `n` coordinates, the embeddings can be organized as the rows of an embedding matrix:

$$
E\in\mathbb{R}^{m\times n}.
$$

> **Important note about embeddings**
>
> An embedding is a compressed, model-dependent numerical representation. It does not preserve every detail of the original object, and high embedding similarity does not by itself guarantee that two statements are factually equivalent.


## 2. Scalars

### Background

Many quantities summarize one aspect of a model or one observation. Examples include a predicted probability, a temperature, a learning rate, or the value of a loss function.

### Definition

A **scalar** is a single number. A scalar may be positive, negative, or zero.

Examples include

$$
0.92,\qquad -1.7,\qquad 5,\qquad \frac{1}{3}.
$$

### AI examples

- A model assigns probability `0.92` to an image containing a dog.
- A training algorithm uses learning rate `0.001`.
- A loss function currently has value `0.37`.

Each is one number, so each is a scalar.


In [ ]:
# Scalars in Python.
predicted_probability = 0.92
learning_rate = 0.001
loss_value = 0.37

print("Predicted probability:", predicted_probability)
print("Learning rate:", learning_rate)
print("Loss value:", loss_value)
print("All three objects are individual numbers.")


## 3. Vectors

### Background

One number is rarely enough to describe a complex object. A song may have features for tempo, energy, and acousticness. A student record may contain several measurements. A word embedding may have hundreds or thousands of learned coordinates.

### Definition

A **vector** is an ordered list of numbers. A real vector with `n` coordinates is written as

$$
\mathbf{x}
=
\begin{bmatrix}
x_1\\
x_2\\
\vdots\\
x_n
\end{bmatrix}
\in \mathbb{R}^n.
$$

The number of coordinates is the vector's **dimension**. The order matters: changing the order changes which feature each number represents.


### Worked example: a three-feature music vector

Suppose a simplified song representation uses these features in this exact order:

1. tempo score;
2. energy score;
3. R&B style score.

The vector

$$
\mathbf{s}
=
\begin{bmatrix}
0.60\\
0.85\\
0.90
\end{bmatrix}
$$

means that the tempo coordinate is `0.60`, the energy coordinate is `0.85`, and the R&B coordinate is `0.90`.

This vector has three coordinates, so

$$
\mathbf{s}\in\mathbb{R}^3.
$$

If we reorder the entries without reordering the feature labels, we change the meaning of the data.


In [ ]:
song_vector = np.array([0.60, 0.85, 0.90])
feature_names = ["tempo", "energy", "R&B style"]

print("Vector:", song_vector)
print("NumPy shape:", song_vector.shape)
print("Number of coordinates:", song_vector.size)
for index, (name, value) in enumerate(zip(feature_names, song_vector), start=1):
    print(f"Coordinate {index}: {name:10s} = {value:.2f}")


## 4. Visualizing a vector as a directed displacement

When a vector has two coordinates, it can be drawn as an arrow from the origin. For example,

$$
\mathbf{v}
=
\begin{bmatrix}
3\\
2
\end{bmatrix}
$$

means move `3` units in the horizontal direction and `2` units in the vertical direction.

The arrow picture is useful for two or three dimensions. AI embeddings often live in far higher dimensions, where we cannot draw the full vector, but the algebraic rules remain the same.


In [ ]:
v_visual = np.array([3.0, 2.0])
plot_two_vectors(v_visual, np.array([0.0, 0.0]),
                 labels=("v = [3, 2]", "origin"),
                 title="A two-dimensional vector")


## 5. Vector addition and scalar multiplication

Before comparing vectors, we need two basic operations.

### Vector addition

Vectors of the same dimension are added coordinate by coordinate:

$$
\begin{bmatrix}
u_1\\u_2\\\vdots\\u_n
\end{bmatrix}
+
\begin{bmatrix}
v_1\\v_2\\\vdots\\v_n
\end{bmatrix}
=
\begin{bmatrix}
u_1+v_1\\u_2+v_2\\\vdots\\u_n+v_n
\end{bmatrix}.
$$

### Scalar multiplication

Multiplying a vector by a scalar multiplies every coordinate:

$$
c
\begin{bmatrix}
v_1\\v_2\\\vdots\\v_n
\end{bmatrix}
=
\begin{bmatrix}
cv_1\\cv_2\\\vdots\\cv_n
\end{bmatrix}.
$$


### Detailed example: adjusting an embedding

Suppose a base embedding and an update direction are

$$
\mathbf{e}
=
\begin{bmatrix}
0.60\\-0.20\\0.40
\end{bmatrix},
\qquad
\mathbf{a}
=
\begin{bmatrix}
0.10\\0.30\\-0.20
\end{bmatrix}.
$$

Apply one-half of the update:

$$
\mathbf{e}_{\mathrm{new}}
=\mathbf{e}+0.5\mathbf{a}.
$$

First multiply every entry of the update by `0.5`:

$$
0.5\mathbf{a}
=
\begin{bmatrix}
0.5(0.10)\\
0.5(0.30)\\
0.5(-0.20)
\end{bmatrix}
=
\begin{bmatrix}
0.05\\0.15\\-0.10
\end{bmatrix}.
$$

Now add corresponding coordinates:

$$
\mathbf{e}_{\mathrm{new}}
=
\begin{bmatrix}
0.60+0.05\\
-0.20+0.15\\
0.40-0.10
\end{bmatrix}
=
\begin{bmatrix}
0.65\\-0.05\\0.30
\end{bmatrix}.
$$


In [ ]:
e = np.array([0.60, -0.20, 0.40])
a = np.array([0.10, 0.30, -0.20])
scaled_update = 0.5 * a
e_new = e + scaled_update

print("Base embedding:       ", e)
print("Half of update:       ", scaled_update)
print("Updated embedding:    ", e_new)


## 6. Matrices

### Background

A dataset usually contains many observations, and each observation may contain many features. A rectangular table organizes this information naturally.

### Definition

An **m by n matrix** is a rectangular array with `m` rows and `n` columns:

$$
A=
\begin{bmatrix}
a_{11} & a_{12} & \cdots & a_{1n}\\
a_{21} & a_{22} & \cdots & a_{2n}\\
\vdots & \vdots & \ddots & \vdots\\
a_{m1} & a_{m2} & \cdots & a_{mn}
\end{bmatrix}.
$$

Its shape is

$$
m\times n.
$$

The first subscript identifies the row; the second identifies the column.


### Worked example: a batch of embeddings

Suppose four short documents are each represented by five features. If one document occupies each row, the embedding matrix has shape

$$
4\times 5.
$$

$$
E=
\begin{bmatrix}
0.9&0.8&0.1&0.0&0.2\\
0.1&0.2&0.9&0.1&0.0\\
0.6&0.7&0.2&0.1&0.3\\
0.0&0.1&0.2&0.9&0.8
\end{bmatrix}.
$$

For example, the entry in row `3`, column `2` is `0.7`. It is the second feature of the third document.


In [ ]:
E = np.array([
    [0.9, 0.8, 0.1, 0.0, 0.2],
    [0.1, 0.2, 0.9, 0.1, 0.0],
    [0.6, 0.7, 0.2, 0.1, 0.3],
    [0.0, 0.1, 0.2, 0.9, 0.8],
])

print("Embedding matrix shape:", E.shape)
print("Python uses zero-based indexing.")
print("E[2, 1] is row 3, column 2:", E[2, 1])

fig, ax = plt.subplots(figsize=(8, 4))
image = ax.imshow(E, cmap="viridis", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(E.shape[1]), [f"feature {j}" for j in range(1, 6)])
ax.set_yticks(range(E.shape[0]), [f"document {i}" for i in range(1, 5)])
ax.set_title("A matrix of document embeddings")
for i in range(E.shape[0]):
    for j in range(E.shape[1]):
        color = "white" if E[i, j] > 0.5 else "black"
        ax.text(j, i, f"{E[i,j]:.1f}", ha="center", va="center", color=color)
fig.colorbar(image, ax=ax, label="feature value")
plt.show()


## 7. Tensors

### Background

Some data require more than two indices. A color image needs a row index, a column index, and a color-channel index. A batch of color images adds a fourth index for the image number.

### Definition used in this course

A **tensor** is a multidimensional numerical array. The number of axes is its **order** or **rank in the array sense**, and the length along each axis forms its **shape**.

> **Vocabulary warning**
>
> Tensor order and matrix rank are different ideas. Matrix rank will be studied in Module 2.

| Object | Number of axes | Example shape |
|---|---:|---|
| Scalar | 0 | `()` |
| Vector | 1 | `(5,)` |
| Matrix | 2 | `(4, 5)` |
| Color image tensor | 3 | `(height, width, channels)` |
| Batch of color images | 4 | `(batch, height, width, channels)` |


### Applied example: a tiny RGB image tensor

An RGB image uses three color channels:

1. red;
2. green;
3. blue.

A tiny image with `4` rows, `4` columns, and `3` color channels has shape

$$
4\times 4\times 3.
$$

One pixel is a length-three vector. For example, the vector

$$
\begin{bmatrix}
1\\0\\0
\end{bmatrix}
$$

represents full red, no green, and no blue.


In [ ]:
# Construct a 4-by-4 RGB image with values between 0 and 1.
tiny_image = np.array([
    [[1,0,0], [1,0.5,0], [1,1,0], [0,1,0]],
    [[0,1,1], [0,0.5,1], [0,0,1], [0.5,0,1]],
    [[1,0,1], [0.8,0.2,0.5], [0.5,0.5,0.5], [1,1,1]],
    [[0.2,0.2,0.2], [0.4,0.2,0.1], [0.1,0.4,0.2], [0.2,0.1,0.4]],
], dtype=float)

print("Tensor shape:", tiny_image.shape)
print("Number of axes:", tiny_image.ndim)
print("Pixel in row 1, column 1:", tiny_image[0, 0])

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
axes[0].imshow(tiny_image)
axes[0].set_title("RGB image")
channel_names = ["red channel", "green channel", "blue channel"]
channel_maps = ["Reds", "Greens", "Blues"]
for channel in range(3):
    axes[channel + 1].imshow(tiny_image[:, :, channel], cmap=channel_maps[channel], vmin=0, vmax=1)
    axes[channel + 1].set_title(channel_names[channel])
for ax in axes:
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
plt.tight_layout()
plt.show()


## 8. How the structures appear in an AI pipeline

| AI quantity | Typical structure | Interpretation |
|---|---|---|
| Learning rate | Scalar | One training-control value |
| One token embedding | Vector | One list of learned features |
| A batch of token embeddings | Matrix | One embedding per row |
| One color image | Third-order tensor | Height, width, and color channel |
| A batch of images | Fourth-order tensor | Batch, height, width, and channel |

The exact axis order varies among software libraries. For that reason, **always inspect the shape before computing**.


### Classroom check: identify the object

Classify each item and state its shape.

1. A confidence score `0.84`.
2. One embedding with `768` coordinates.
3. A batch of `32` embeddings, each with `768` coordinates, stored one embedding per row.
4. A batch of `16` RGB images, each `224` pixels high and `224` pixels wide, stored as `(batch, height, width, channel)`.


In [ ]:
reveal_markdown("Reveal instructor solution", r'''
1. The confidence score is a **scalar** with shape `()`.
2. The embedding is a **vector** with shape `(768,)`.
3. The batch is a **matrix** with shape `(32, 768)`.
4. The image batch is a **fourth-order tensor** with shape `(16, 224, 224, 3)`.

The final coordinate is `3` because an RGB image has red, green, and blue channels.
''')


> **Common mistakes in Section 1**
>
> - Treating a vector as an unordered collection. Coordinate order carries meaning.
> - Confusing the number of entries with the numerical values of the entries.
> - Calling every array a matrix. A matrix has exactly two axes.
> - Confusing tensor order with matrix rank.
> - Forgetting that NumPy indices begin at zero.
> - Assuming every library stores image channels in the same axis position.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Scalar | One number |
| Vector | An ordered one-dimensional list of numbers |
| Dimension | Number of coordinates in a vector |
| Matrix | A rectangular two-dimensional array |
| Tensor | A multidimensional numerical array in the ML convention |
| Shape | Length of the array along each axis |
| Embedding | A numerical vector representing an object such as text, audio, or an image |


## Section 1 worksheet

1. Classify each object as a scalar, vector, matrix, or tensor:
   - one model loss value;
   - the five features of one patient;
   - the five features of `100` patients;
   - a batch of grayscale videos indexed by video, time, height, and width.
2. A dataset matrix has `40` rows and `6` columns, with one person per row. How many people and how many features are represented?
3. Let

$$
\mathbf{u}=
\begin{bmatrix}
2\\-1\\4
\end{bmatrix},
\qquad
\mathbf{v}=
\begin{bmatrix}
-3\\5\\2
\end{bmatrix}.
$$

Compute

$$
2\mathbf{u}+\mathbf{v}.
$$

4. An RGB tensor has shape `(128, 128, 3)`. Explain every number in the shape.


In [ ]:
reveal_markdown("Open Section 1 solutions", r'''
### 1. Classification

- One loss value is a **scalar**.
- Five features for one patient form a **vector** of shape `(5,)`.
- Five features for `100` patients form a **matrix** of shape `(100, 5)` if one patient is stored per row.
- The video batch is a **fourth-order tensor** because four indices are needed.

### 2. Dataset matrix

There are `40` people and `6` features per person.

### 3. Vector calculation

First compute the scalar multiple:

$$
2\mathbf{u}
=
2
\begin{bmatrix}
2\\-1\\4
\end{bmatrix}
=
\begin{bmatrix}
4\\-2\\8
\end{bmatrix}.
$$

Then add corresponding coordinates:

$$
2\mathbf{u}+\mathbf{v}
=
\begin{bmatrix}
4\\-2\\8
\end{bmatrix}
+
\begin{bmatrix}
-3\\5\\2
\end{bmatrix}
=
\begin{bmatrix}
1\\3\\10
\end{bmatrix}.
$$

### 4. RGB shape

The image has `128` rows of pixels, `128` columns of pixels, and `3` color channels.
''')


---

# Section 2: Dot Products and Orthogonality

## Learning objectives

Students should be able to compute a dot product, interpret the contribution from each coordinate, connect the dot product to angle, identify orthogonal vectors, compute a projection, and interpret a linear neuron as a dot product plus a bias.


## 1. Why multiply matching coordinates?

Suppose two vectors describe the same ordered features. A natural comparison pairs feature `1` with feature `1`, feature `2` with feature `2`, and so on. The dot product multiplies those matched pairs and then adds their contributions.

Large positive matched products increase the result. Large negative matched products decrease it. A zero coordinate contributes nothing.


## 2. Definition of the dot product

For vectors of the same dimension,

$$
\mathbf{u}=
\begin{bmatrix}
u_1\\u_2\\\vdots\\u_n
\end{bmatrix},
\qquad
\mathbf{v}=
\begin{bmatrix}
v_1\\v_2\\\vdots\\v_n
\end{bmatrix},
$$

their **dot product** or **inner product** is

$$
\mathbf{u}\cdot\mathbf{v}
=u_1v_1+u_2v_2+\cdots+u_nv_n
=\sum_{i=1}^{n}u_iv_i.
$$

Matrix notation writes the same calculation as

$$
\mathbf{u}^{T}\mathbf{v}.
$$

The input objects are vectors, but the output is one number: the dot product is a scalar.


### Detailed worked example

Let

$$
\mathbf{u}=
\begin{bmatrix}
2\\-1\\3
\end{bmatrix},
\qquad
\mathbf{v}=
\begin{bmatrix}
4\\0.5\\-2
\end{bmatrix}.
$$

**Step 1: match coordinates.**

$$
(2,4),\qquad(-1,0.5),\qquad(3,-2).
$$

**Step 2: multiply within each pair.**

$$
2(4)=8,
$$

$$
(-1)(0.5)=-0.5,
$$

$$
3(-2)=-6.
$$

**Step 3: add all contributions.**

$$
\mathbf{u}\cdot\mathbf{v}
=8+(-0.5)+(-6)
=1.5.
$$


In [ ]:
u_dot = np.array([2.0, -1.0, 3.0])
v_dot = np.array([4.0, 0.5, -2.0])
contributions = u_dot * v_dot

print("Coordinate products:", contributions)
print("Sum of products:", contributions.sum())
print("NumPy dot product:", np.dot(u_dot, v_dot))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2ca02c" if value >= 0 else "#d62728" for value in contributions]
ax.bar(["feature 1", "feature 2", "feature 3"], contributions, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("contribution to dot product")
ax.set_title("Every coordinate contributes to the final dot product")
for i, value in enumerate(contributions):
    ax.text(i, value + (0.25 if value >= 0 else -0.45), f"{value:g}", ha="center")
plt.show()


## 3. Dot-product properties

For same-dimensional vectors and any scalar `c`, the following properties hold.

### Symmetry

$$
\mathbf{u}\cdot\mathbf{v}=\mathbf{v}\cdot\mathbf{u}.
$$

This follows because ordinary multiplication is commutative:

$$
u_iv_i=v_iu_i
$$

for every coordinate.

### Distributivity

$$
\mathbf{u}\cdot(\mathbf{v}+\mathbf{w})
=\mathbf{u}\cdot\mathbf{v}+\mathbf{u}\cdot\mathbf{w}.
$$

Entry by entry,

$$
\sum_{i=1}^{n}u_i(v_i+w_i)
=\sum_{i=1}^{n}(u_iv_i+u_iw_i)
=\sum_{i=1}^{n}u_iv_i+\sum_{i=1}^{n}u_iw_i.
$$

### Compatibility with scalar multiplication

$$
(c\mathbf{u})\cdot\mathbf{v}
=c(\mathbf{u}\cdot\mathbf{v}).
$$

### Self-dot product

$$
\mathbf{u}\cdot\mathbf{u}
=u_1^2+u_2^2+\cdots+u_n^2\geq 0.
$$

The self-dot product equals zero only when every coordinate is zero.


> **Dimension rule**
>
> The standard dot product requires the two vectors to have the same number of coordinates. A length-three vector and a length-four vector do not have a standard dot product because one coordinate would have no matched partner.


## 4. Geometric meaning of the dot product

The dot product also measures directional alignment:

$$
\mathbf{u}\cdot\mathbf{v}
=\lVert\mathbf{u}\rVert_2
\lVert\mathbf{v}\rVert_2
\cos(\theta),
$$

where `theta` is the angle between the vectors and the L2 norm measures length.

Therefore:

| Angle relation | Sign of dot product | Directional meaning |
|---|---:|---|
| Less than 90 degrees | Positive | broadly aligned |
| Exactly 90 degrees | Zero | perpendicular |
| Greater than 90 degrees | Negative | broadly opposed |

The sizes of the vectors also affect the dot product. Section 4 will remove that size effect using cosine similarity.


In [ ]:
# Interactive exploration of dot product, angle, and vector length.
def dot_angle_demo(ux=2.0, uy=1.0, vx=1.0, vy=2.0):
    u = np.array([ux, uy], dtype=float)
    v = np.array([vx, vy], dtype=float)
    dot = float(np.dot(u, v))
    norm_product = np.linalg.norm(u) * np.linalg.norm(v)
    if np.isclose(norm_product, 0):
        angle_text = "undefined because one vector is zero"
    else:
        cosine = np.clip(dot / norm_product, -1.0, 1.0)
        angle_text = f"{np.degrees(np.arccos(cosine)):.2f} degrees"
    print(f"dot product = {dot:.3f}")
    print("angle =", angle_text)
    plot_two_vectors(u, v, title="Dot product and angle")

if WIDGETS_AVAILABLE:
    interact(
        dot_angle_demo,
        ux=FloatSlider(value=2, min=-4, max=4, step=0.5, description="u₁"),
        uy=FloatSlider(value=1, min=-4, max=4, step=0.5, description="u₂"),
        vx=FloatSlider(value=1, min=-4, max=4, step=0.5, description="v₁"),
        vy=FloatSlider(value=2, min=-4, max=4, step=0.5, description="v₂"),
    )
else:
    dot_angle_demo()


## 5. Orthogonality

### Definition

Two vectors are **orthogonal** when their dot product is zero:

$$
\mathbf{u}\perp\mathbf{v}
\quad\Longleftrightarrow\quad
\mathbf{u}\cdot\mathbf{v}=0.
$$

For nonzero vectors, orthogonality means the angle between them is `90` degrees. The zero vector has dot product zero with every vector, although it has no geometric direction.


### Detailed orthogonality check

Let

$$
\mathbf{u}=
\begin{bmatrix}
2\\1\\-1
\end{bmatrix},
\qquad
\mathbf{v}=
\begin{bmatrix}
1\\-1\\1
\end{bmatrix}.
$$

Compute the dot product:

$$
\mathbf{u}\cdot\mathbf{v}
=2(1)+1(-1)+(-1)(1).
$$

Evaluate each product:

$$
2(1)=2,\qquad 1(-1)=-1,\qquad (-1)(1)=-1.
$$

Add:

$$
2-1-1=0.
$$

Therefore,

$$
\mathbf{u}\perp\mathbf{v}.
$$


In [ ]:
u_orth = np.array([2, 1, -1])
v_orth = np.array([1, -1, 1])
print("u dot v =", np.dot(u_orth, v_orth))

# A two-dimensional orthogonal pair for visualization.
plot_two_vectors([2, 1], [-1, 2],
                 title="A visible orthogonal pair: [2,1] and [-1,2]")


## 6. Projection: separating aligned and orthogonal parts

### Background

Suppose we want the part of a vector that points along a chosen direction. Projection extracts that aligned component. The leftover part is perpendicular to the chosen direction.

### Definition

For a nonzero vector `u`, the projection of `v` onto `u` is

$$
\operatorname{proj}_{\mathbf{u}}(\mathbf{v})
=
\frac{\mathbf{v}\cdot\mathbf{u}}
{\mathbf{u}\cdot\mathbf{u}}
\mathbf{u}.
$$

The fraction is a scalar. It tells us how much of the direction `u` is present in `v`.


### Detailed projection example

Let

$$
\mathbf{v}=
\begin{bmatrix}
4\\2
\end{bmatrix},
\qquad
\mathbf{u}=
\begin{bmatrix}
1\\2
\end{bmatrix}.
$$

First compute the numerator:

$$
\mathbf{v}\cdot\mathbf{u}
=4(1)+2(2)
=4+4
=8.
$$

Next compute the denominator:

$$
\mathbf{u}\cdot\mathbf{u}
=1^2+2^2
=1+4
=5.
$$

Therefore,

$$
\operatorname{proj}_{\mathbf{u}}(\mathbf{v})
=\frac{8}{5}
\begin{bmatrix}
1\\2
\end{bmatrix}
=
\begin{bmatrix}
\frac{8}{5}\\\frac{16}{5}
\end{bmatrix}
=
\begin{bmatrix}
1.6\\3.2
\end{bmatrix}.
$$

The residual is

$$
\mathbf{r}
=\mathbf{v}-\operatorname{proj}_{\mathbf{u}}(\mathbf{v})
=
\begin{bmatrix}
4\\2
\end{bmatrix}
-
\begin{bmatrix}
1.6\\3.2
\end{bmatrix}
=
\begin{bmatrix}
2.4\\-1.2
\end{bmatrix}.
$$

Verify orthogonality:

$$
\mathbf{r}\cdot\mathbf{u}
=2.4(1)+(-1.2)(2)
=2.4-2.4
=0.
$$


In [ ]:
v_proj = np.array([4.0, 2.0])
u_proj = np.array([1.0, 2.0])
coefficient = np.dot(v_proj, u_proj) / np.dot(u_proj, u_proj)
projection = coefficient * u_proj
residual = v_proj - projection

print("Projection coefficient:", coefficient)
print("Projection:", projection)
print("Residual:", residual)
print("Residual dot u:", np.dot(residual, u_proj))

fig, ax = plt.subplots(figsize=(7, 6))
ax.quiver(0, 0, v_proj[0], v_proj[1], angles="xy", scale_units="xy", scale=1,
          color="#1f77b4", width=0.012, label="v")
ax.quiver(0, 0, projection[0], projection[1], angles="xy", scale_units="xy", scale=1,
          color="#2ca02c", width=0.012, label="projection onto u")
ax.quiver(projection[0], projection[1], residual[0], residual[1],
          angles="xy", scale_units="xy", scale=1,
          color="#d62728", width=0.012, label="orthogonal residual")
line_t = np.linspace(-1, 3, 100)
ax.plot(line_t * u_proj[0], line_t * u_proj[1], "--", color="gray", label="direction of u")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect("equal")
ax.set_title("Vector = projection + orthogonal residual")
ax.legend(loc="upper left")
plt.show()


## 7. Direct AI application: a linear neuron

A basic neuron first forms a weighted sum of input features:

$$
z=\mathbf{w}\cdot\mathbf{x}+b.
$$

The vector `x` contains input features, the vector `w` contains learned weights, and `b` is a scalar bias.

Let

$$
\mathbf{x}=
\begin{bmatrix}
0.6\\-1.2\\0.5
\end{bmatrix},
\qquad
\mathbf{w}=
\begin{bmatrix}
1.5\\-0.5\\2.0
\end{bmatrix},
\qquad
b=-0.3.
$$

Compute the weighted feature contributions:

$$
1.5(0.6)=0.9,
$$

$$
(-0.5)(-1.2)=0.6,
$$

$$
2.0(0.5)=1.0.
$$

Thus,

$$
\mathbf{w}\cdot\mathbf{x}
=0.9+0.6+1.0
=2.5,
$$

and

$$
z=2.5-0.3=2.2.
$$

The next layer may apply a nonlinear activation function, but the pre-activation score begins with this dot product.


In [ ]:
x_neuron = np.array([0.6, -1.2, 0.5])
w_neuron = np.array([1.5, -0.5, 2.0])
b_neuron = -0.3
feature_contributions = x_neuron * w_neuron
z = np.dot(w_neuron, x_neuron) + b_neuron

print("Feature contributions:", feature_contributions)
print("Weighted sum:", feature_contributions.sum())
print("Bias:", b_neuron)
print("Pre-activation z:", z)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["feature 1", "feature 2", "feature 3", "bias"],
              [*feature_contributions, b_neuron],
              color=["#4c78a8", "#4c78a8", "#4c78a8", "#f58518"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("contribution")
ax.set_title(f"Neuron contributions add to z = {z:.1f}")
plt.show()


### Classroom check

Let

$$
\mathbf{a}=
\begin{bmatrix}
1\\2\\-1
\end{bmatrix},
\qquad
\mathbf{b}=
\begin{bmatrix}
3\\0\\3
\end{bmatrix}.
$$

1. Compute the dot product.
2. Are the vectors orthogonal?
3. If the second vector is doubled, what happens to the dot product?


In [ ]:
reveal_markdown("Reveal instructor solution", r'''
### 1. Dot product

$$
\mathbf{a}\cdot\mathbf{b}
=1(3)+2(0)+(-1)(3)
=3+0-3
=0.
$$

### 2. Orthogonality

The dot product is zero, so the vectors are orthogonal.

### 3. Doubling the second vector

Using compatibility with scalar multiplication,

$$
\mathbf{a}\cdot(2\mathbf{b})
=2(\mathbf{a}\cdot\mathbf{b})
=2(0)
=0.
$$

The new vector remains orthogonal to the first vector.
''')


> **Common mistakes in Section 2**
>
> - Adding coordinate products incorrectly after multiplying them.
> - Computing a dot product for vectors with different dimensions.
> - Believing that a zero dot product means one vector must be zero.
> - Forgetting the denominator in the projection formula.
> - Using the geometric angle formula when one vector is zero.
> - Treating the dot product as a vector instead of a scalar.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Dot product | Sum of products of corresponding coordinates |
| Inner product | Another name for the standard dot product in this setting |
| Orthogonal | Having dot product zero |
| Projection | Component of one vector along another nonzero vector |
| Residual | Difference between the original vector and its projection |
| Weight vector | Learned feature coefficients used by a model |
| Bias | Scalar offset added to a weighted sum |


## Section 2 worksheet

1. Compute the dot product:

$$
\begin{bmatrix}
2\\-3\\4
\end{bmatrix}
\cdot
\begin{bmatrix}
-1\\5\\2
\end{bmatrix}.
$$

2. Find the value of `k` that makes the following vectors orthogonal:

$$
\begin{bmatrix}
1\\2\\k
\end{bmatrix},
\qquad
\begin{bmatrix}
4\\-1\\2
\end{bmatrix}.
$$

3. Project

$$
\mathbf{v}=
\begin{bmatrix}
3\\4
\end{bmatrix}
$$

onto

$$
\mathbf{u}=
\begin{bmatrix}
1\\0
\end{bmatrix}.
$$

4. A neuron has input vector `[2, -1]`, weight vector `[0.5, 3]`, and bias `1`. Compute its pre-activation score.


In [ ]:
reveal_markdown("Open Section 2 solutions", r'''
### 1. Dot product

$$
2(-1)+(-3)(5)+4(2)
=-2-15+8
=-9.
$$

### 2. Orthogonality parameter

Set the dot product equal to zero:

$$
1(4)+2(-1)+k(2)=0.
$$

$$
4-2+2k=0.
$$

$$
2+2k=0.
$$

$$
2k=-2.
$$

$$
k=-1.
$$

### 3. Projection

$$
\mathbf{v}\cdot\mathbf{u}=3(1)+4(0)=3,
$$

$$
\mathbf{u}\cdot\mathbf{u}=1^2+0^2=1.
$$

Therefore,

$$
\operatorname{proj}_{\mathbf{u}}(\mathbf{v})
=\frac{3}{1}
\begin{bmatrix}
1\\0
\end{bmatrix}
=
\begin{bmatrix}
3\\0
\end{bmatrix}.
$$

### 4. Neuron score

$$
z=0.5(2)+3(-1)+1=1-3+1=-1.
$$
''')


---

# Section 3: L1 Norm, L2 Norm, Vector Distance, and Regularization

## Learning objectives

Students should be able to compute L1 and L2 norms, explain what each norm measures, compute L1 and L2 distances, visualize their geometry, and explain why AI training objectives add norm-based penalties.


## 1. Why a vector needs a measure of size

A vector can represent a displacement, a collection of model weights, or an embedding. In each setting, we may ask how large the vector is.

There is more than one sensible definition of size. Different norms emphasize different aspects of a vector. The L1 norm totals absolute coordinate magnitudes, while the L2 norm measures ordinary Euclidean length.


## 2. What is a norm?

A **norm** assigns a nonnegative size to every vector. A norm must satisfy three essential rules.

### Positivity and definiteness

$$
\lVert\mathbf{x}\rVert\geq 0,
$$

and

$$
\lVert\mathbf{x}\rVert=0
\quad\Longleftrightarrow\quad
\mathbf{x}=\mathbf{0}.
$$

### Absolute homogeneity

$$
\lVert c\mathbf{x}\rVert
=|c|\lVert\mathbf{x}\rVert.
$$

### Triangle inequality

$$
\lVert\mathbf{x}+\mathbf{y}\rVert
\leq
\lVert\mathbf{x}\rVert+\lVert\mathbf{y}\rVert.
$$

The triangle inequality says that traveling directly is no longer than traveling through an intermediate displacement.


## 3. L1 norm

The **L1 norm** adds absolute coordinate values:

$$
\lVert\mathbf{x}\rVert_1
=|x_1|+|x_2|+\cdots+|x_n|
=\sum_{i=1}^{n}|x_i|.
$$

Absolute values are essential. Without them, positive and negative entries could cancel even when the vector is far from zero.


## 4. L2 norm

The **L2 norm**, also called the Euclidean norm, is

$$
\lVert\mathbf{x}\rVert_2
=\sqrt{x_1^2+x_2^2+\cdots+x_n^2}
=\sqrt{\sum_{i=1}^{n}x_i^2}.
$$

It is connected directly to the dot product:

$$
\lVert\mathbf{x}\rVert_2
=\sqrt{\mathbf{x}\cdot\mathbf{x}}.
$$


### Detailed comparison example

Let

$$
\mathbf{x}=
\begin{bmatrix}
3\\-4\\12
\end{bmatrix}.
$$

For the L1 norm, take absolute values and add:

$$
\lVert\mathbf{x}\rVert_1
=|3|+|-4|+|12|
=3+4+12
=19.
$$

For the L2 norm, square, add, and take a square root:

$$
\lVert\mathbf{x}\rVert_2
=\sqrt{3^2+(-4)^2+12^2}.
$$

$$
=\sqrt{9+16+144}.
$$

$$
=\sqrt{169}.
$$

$$
=13.
$$

Both norms measure the same vector, but they answer the size question differently.


In [ ]:
x_norm = np.array([3.0, -4.0, 12.0])
l1 = np.linalg.norm(x_norm, ord=1)
l2 = np.linalg.norm(x_norm, ord=2)

print("Absolute coordinate values:", np.abs(x_norm))
print("Squared coordinate values: ", x_norm**2)
print("L1 norm:", l1)
print("L2 norm:", l2)
print("sqrt(x dot x):", np.sqrt(np.dot(x_norm, x_norm)))


## 5. Comparing L1 and L2 geometry

In two dimensions, all vectors with L1 norm equal to `1` satisfy

$$
|x_1|+|x_2|=1.
$$

This boundary is a diamond.

All vectors with L2 norm equal to `1` satisfy

$$
x_1^2+x_2^2=1.
$$

This boundary is a circle. The corners of the L1 diamond help explain why L1 regularization often produces exact zeros.


In [ ]:
theta = np.linspace(0, 2*np.pi, 600)
l2_x = np.cos(theta)
l2_y = np.sin(theta)

l1_x = np.array([1, 0, -1, 0, 1])
l1_y = np.array([0, 1, 0, -1, 0])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].plot(l1_x, l1_y, color="#d62728", linewidth=3)
axes[0].fill(l1_x, l1_y, color="#d62728", alpha=0.12)
axes[0].set_title("L1 unit ball: |x₁| + |x₂| ≤ 1")
axes[1].plot(l2_x, l2_y, color="#1f77b4", linewidth=3)
axes[1].fill(l2_x, l2_y, color="#1f77b4", alpha=0.12)
axes[1].set_title("L2 unit ball: x₁² + x₂² ≤ 1")
for ax in axes:
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-1.25, 1.25)
    ax.set_aspect("equal")
    ax.set_xlabel("x₁")
    ax.set_ylabel("x₂")
plt.tight_layout()
plt.show()


In [ ]:
# Interactive norm explorer.
def norm_explorer(x1=3.0, x2=4.0):
    x = np.array([x1, x2], dtype=float)
    print(f"vector = {x}")
    print(f"L1 norm = |{x1:g}| + |{x2:g}| = {np.linalg.norm(x, 1):.3f}")
    print(f"L2 norm = sqrt(({x1:g})² + ({x2:g})²) = {np.linalg.norm(x, 2):.3f}")
    plot_two_vectors(x, [0, 0], labels=("x", "origin"), title="Vector whose norms are being compared")

if WIDGETS_AVAILABLE:
    interact(
        norm_explorer,
        x1=FloatSlider(value=3, min=-6, max=6, step=0.5, description="x₁"),
        x2=FloatSlider(value=4, min=-6, max=6, step=0.5, description="x₂"),
    )
else:
    norm_explorer()


## 6. Vector distance

### Background

To measure how far two vectors are from each other, first subtract them. The difference vector points from one vector to the other. A norm then measures the size of that difference.

For any norm, the induced distance is

$$
d(\mathbf{x},\mathbf{y})
=\lVert\mathbf{x}-\mathbf{y}\rVert.
$$

The L1 and L2 distances are

$$
d_1(\mathbf{x},\mathbf{y})
=\lVert\mathbf{x}-\mathbf{y}\rVert_1,
$$

$$
d_2(\mathbf{x},\mathbf{y})
=\lVert\mathbf{x}-\mathbf{y}\rVert_2.
$$


### Detailed distance example

Let

$$
\mathbf{x}=
\begin{bmatrix}
2\\-1\\4
\end{bmatrix},
\qquad
\mathbf{y}=
\begin{bmatrix}
-1\\3\\2
\end{bmatrix}.
$$

First subtract corresponding coordinates:

$$
\mathbf{x}-\mathbf{y}
=
\begin{bmatrix}
2-(-1)\\
-1-3\\
4-2
\end{bmatrix}
=
\begin{bmatrix}
3\\-4\\2
\end{bmatrix}.
$$

The L1 distance is

$$
d_1(\mathbf{x},\mathbf{y})
=|3|+|-4|+|2|
=3+4+2
=9.
$$

The L2 distance is

$$
d_2(\mathbf{x},\mathbf{y})
=\sqrt{3^2+(-4)^2+2^2}
=\sqrt{9+16+4}
=\sqrt{29}
\approx 5.385.
$$


In [ ]:
x_dist = np.array([2.0, -1.0, 4.0])
y_dist = np.array([-1.0, 3.0, 2.0])
difference = x_dist - y_dist

print("x - y:", difference)
print("L1 distance:", np.linalg.norm(difference, 1))
print("L2 distance:", np.linalg.norm(difference, 2))
print("Symmetry check, ||y-x||₂:", np.linalg.norm(y_dist - x_dist, 2))


### Visual example: direct distance versus grid distance

Suppose two points are

$$
A=(1,1),\qquad B=(5,4).
$$

The coordinate difference is

$$
B-A=(4,3).
$$

The L1 distance is

$$
|4|+|3|=7,
$$

while the L2 distance is

$$
\sqrt{4^2+3^2}=\sqrt{25}=5.
$$

L1 distance resembles travel along horizontal and vertical streets. L2 distance is the direct straight-line route.


In [ ]:
A_point = np.array([1.0, 1.0])
B_point = np.array([5.0, 4.0])

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot([A_point[0], B_point[0]], [A_point[1], B_point[1]],
        color="#1f77b4", linewidth=3, label="L2 direct path: length 5")
ax.plot([A_point[0], B_point[0], B_point[0]],
        [A_point[1], A_point[1], B_point[1]],
        color="#d62728", linewidth=3, linestyle="--", label="one L1 path: length 7")
ax.scatter([A_point[0], B_point[0]], [A_point[1], B_point[1]], s=90, color="black")
ax.text(A_point[0]-0.2, A_point[1]-0.35, "A")
ax.text(B_point[0]+0.1, B_point[1]+0.1, "B")
ax.set_xlim(0, 6)
ax.set_ylim(0, 5)
ax.set_aspect("equal")
ax.set_xlabel("horizontal coordinate")
ax.set_ylabel("vertical coordinate")
ax.set_title("Two different notions of distance")
ax.legend()
plt.show()


> **Feature-scale warning**
>
> Distance depends on the numerical scale of each coordinate. If one feature is measured in thousands and another lies between zero and one, the large-scale feature may dominate. Standardization or another meaningful scaling step is often necessary before comparing data vectors.


## 7. Direct AI application: regularization

### Background

An AI model can fit training data extremely well by using complicated or very large weights. That fit may not transfer to new data, a problem called **overfitting**.

Regularization adds a penalty to the training objective:

$$
\text{regularized objective}
=\text{data-fitting loss}+\text{weight penalty}.
$$

Two common choices are

$$
J_{L1}(\mathbf{w})
=L(\mathbf{w})+\lambda\lVert\mathbf{w}\rVert_1,
$$

and

$$
J_{L2}(\mathbf{w})
=L(\mathbf{w})+\lambda\lVert\mathbf{w}\rVert_2^2.
$$

The scalar `lambda` controls penalty strength. Larger values place more emphasis on keeping weights small.

> **Convention note**
>
> What practitioners call *L2 regularization* usually uses the **squared** L2 norm. Squaring makes differentiation especially convenient.


### Detailed one-coordinate L2 derivation

Suppose the unregularized preferred weight is `a`, and consider

$$
J_{L2}(w)
=\frac{1}{2}(w-a)^2+\lambda w^2.
$$

Differentiate:

$$
J_{L2}'(w)
=\frac{1}{2}\cdot 2(w-a)+2\lambda w.
$$

Therefore,

$$
J_{L2}'(w)=w-a+2\lambda w.
$$

Set the derivative equal to zero:

$$
w-a+2\lambda w=0.
$$

Collect the terms containing `w`:

$$
(1+2\lambda)w=a.
$$

Thus,

$$
w_{L2}=\frac{a}{1+2\lambda}.
$$

The weight is smoothly shrunk toward zero, but it is not usually exactly zero.


### Detailed one-coordinate L1 derivation

Now consider

$$
J_{L1}(w)
=\frac{1}{2}(w-a)^2+\lambda|w|.
$$

Because the absolute value changes formula at zero, analyze cases.

**Case 1: positive weight.** If `w` is positive, then

$$
|w|=w.
$$

The derivative is

$$
J_{L1}'(w)=w-a+\lambda.
$$

Setting it to zero gives

$$
w=a-\lambda.
$$

This is positive only when `a` is greater than `lambda`.

**Case 2: negative weight.** If `w` is negative, then

$$
|w|=-w.
$$

The derivative is

$$
J_{L1}'(w)=w-a-\lambda.
$$

Setting it to zero gives

$$
w=a+\lambda.
$$

This is negative only when `a` is less than negative `lambda`.

**Case 3: the preferred value is close to zero.** If

$$
|a|\leq\lambda,
$$

the minimum occurs at

$$
w=0.
$$

Combining the cases gives the soft-thresholding rule

$$
w_{L1}
=\operatorname{sign}(a)\max(|a|-\lambda,0).
$$

This exact-zero region is the source of L1 sparsity.


### Vector regularization example

Let the unregularized preferred weight vector be

$$
\mathbf{a}=
\begin{bmatrix}
2.8\\0.7
\end{bmatrix},
$$

and let

$$
\lambda=1.
$$

For L1 regularization, apply soft-thresholding coordinate by coordinate:

$$
w_1=\max(2.8-1,0)=1.8,
$$

$$
w_2=\max(0.7-1,0)=0.
$$

Therefore,

$$
\mathbf{w}_{L1}=
\begin{bmatrix}
1.8\\0
\end{bmatrix}.
$$

For squared-L2 regularization,

$$
\mathbf{w}_{L2}
=\frac{1}{1+2(1)}
\begin{bmatrix}
2.8\\0.7
\end{bmatrix}
=\frac{1}{3}
\begin{bmatrix}
2.8\\0.7
\end{bmatrix}
=
\begin{bmatrix}
0.9333\\0.2333
\end{bmatrix}.
$$

L1 sets the smaller coordinate exactly to zero. L2 keeps both coordinates but shrinks both.


In [ ]:
a_reg = np.array([2.8, 0.7])
lambdas = np.linspace(0, 3, 301)

def soft_threshold(values, lam):
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.maximum(np.abs(values) - lam, 0.0)

l1_path = np.array([soft_threshold(a_reg, lam) for lam in lambdas])
l2_path = np.array([a_reg / (1 + 2*lam) for lam in lambdas])

print("At lambda = 1")
print("L1 solution:", soft_threshold(a_reg, 1.0))
print("L2 solution:", a_reg / 3.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for coordinate in range(2):
    axes[0].plot(lambdas, l1_path[:, coordinate], label=f"weight {coordinate+1}")
    axes[1].plot(lambdas, l2_path[:, coordinate], label=f"weight {coordinate+1}")
axes[0].set_title("L1: weights can become exactly zero")
axes[1].set_title("Squared L2: smooth shrinkage")
for ax in axes:
    ax.axvline(1, color="gray", linestyle="--", label="lambda = 1")
    ax.set_xlabel("regularization strength lambda")
    ax.set_ylabel("regularized weight")
    ax.legend()
plt.tight_layout()
plt.show()


### Practical interpretation

| Penalty | Typical effect | Useful intuition |
|---|---|---|
| L1 | Some weights can become exactly zero | encourages sparse models or feature selection |
| Squared L2 | Weights shrink smoothly | discourages any one weight from becoming too large |

Regularization does not guarantee better performance. The penalty strength must be selected using validation data or another principled procedure.


### Classroom check

For

$$
\mathbf{x}=
\begin{bmatrix}
-2\\5\\-1
\end{bmatrix},
$$

1. compute the L1 norm;
2. compute the L2 norm;
3. compute the L1-regularized soft-thresholded vector when `lambda` equals `2`.


In [ ]:
reveal_markdown("Reveal instructor solution", r'''
### 1. L1 norm

$$
\lVert\mathbf{x}\rVert_1
=|-2|+|5|+|-1|
=2+5+1
=8.
$$

### 2. L2 norm

$$
\lVert\mathbf{x}\rVert_2
=\sqrt{(-2)^2+5^2+(-1)^2}
=\sqrt{4+25+1}
=\sqrt{30}.
$$

### 3. Soft thresholding with lambda equal to 2

Apply the rule to each coordinate:

$$
-2\longmapsto 0,
$$

$$
5\longmapsto 3,
$$

$$
-1\longmapsto 0.
$$

Therefore, the regularized vector is

$$
\begin{bmatrix}
0\\3\\0
\end{bmatrix}.
$$
''')


> **Common mistakes in Section 3**
>
> - Omitting absolute values in the L1 norm.
> - Forgetting the square root in the L2 norm.
> - Confusing a norm with a distance. A norm measures one vector; a distance measures the difference between two vectors.
> - Subtracting vectors with mismatched feature orders.
> - Assuming L1 distance must equal L2 distance.
> - Saying that L2 regularization makes weights exactly zero in general.
> - Forgetting that feature scaling changes distance and regularization effects.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Norm | A function that measures vector size and satisfies norm rules |
| L1 norm | Sum of absolute coordinate values |
| L2 norm | Square root of the sum of squared coordinates |
| Distance | Norm of the difference between two vectors |
| Regularization | Addition of a complexity penalty to a training objective |
| Sparsity | Presence of many exact zeros |
| Regularization strength | Scalar controlling the penalty size |


## Section 3 worksheet

1. For

$$
\mathbf{v}=
\begin{bmatrix}
1\\-2\\2
\end{bmatrix},
$$

compute both the L1 and L2 norms.

2. Compute the L1 and L2 distances between

$$
\mathbf{x}=
\begin{bmatrix}
1\\4
\end{bmatrix},
\qquad
\mathbf{y}=
\begin{bmatrix}
5\\1
\end{bmatrix}.
$$

3. If

$$
\lVert\mathbf{u}\rVert_2=5,
$$

compute

$$
\lVert-3\mathbf{u}\rVert_2.
$$

4. Use soft thresholding with `lambda` equal to `1.5` on

$$
\begin{bmatrix}
3\\-1\\-4
\end{bmatrix}.
$$


In [ ]:
reveal_markdown("Open Section 3 solutions", r'''
### 1. Norms

$$
\lVert\mathbf{v}\rVert_1
=|1|+|-2|+|2|
=1+2+2
=5.
$$

$$
\lVert\mathbf{v}\rVert_2
=\sqrt{1^2+(-2)^2+2^2}
=\sqrt{1+4+4}
=3.
$$

### 2. Distances

First subtract:

$$
\mathbf{x}-\mathbf{y}
=
\begin{bmatrix}
1-5\\4-1
\end{bmatrix}
=
\begin{bmatrix}
-4\\3
\end{bmatrix}.
$$

Then

$$
d_1=|-4|+|3|=7,
$$

and

$$
d_2=\sqrt{(-4)^2+3^2}=\sqrt{16+9}=5.
$$

### 3. Scalar multiple

Absolute homogeneity gives

$$
\lVert-3\mathbf{u}\rVert_2
=|-3|\lVert\mathbf{u}\rVert_2
=3(5)
=15.
$$

### 4. Soft thresholding

$$
3\longmapsto 3-1.5=1.5,
$$

$$
-1\longmapsto 0,
$$

$$
-4\longmapsto -(4-1.5)=-2.5.
$$

The result is

$$
\begin{bmatrix}
1.5\\0\\-2.5
\end{bmatrix}.
$$
''')


---

# Section 4: Cosine Similarity and Embedding Retrieval

## Learning objectives

Students should be able to compute cosine similarity, interpret its range, explain its behavior under scaling, identify the zero-vector exception, compare dot-product and cosine rankings, and carry out a small embedding-retrieval workflow.


## 1. Why direction can matter more than magnitude

The dot product depends on both direction and vector size. A very long vector can have a large dot product even when its direction is not the best match.

For embeddings, we often want to compare the pattern of coordinates rather than their overall scale. Cosine similarity divides out the two lengths, leaving a directional comparison.


## 2. Definition of cosine similarity

For nonzero vectors of the same dimension,

$$
\operatorname{cosim}(\mathbf{u},\mathbf{v})
=
\frac{\mathbf{u}\cdot\mathbf{v}}
{\lVert\mathbf{u}\rVert_2\lVert\mathbf{v}\rVert_2}.
$$

Because

$$
\mathbf{u}\cdot\mathbf{v}
=\lVert\mathbf{u}\rVert_2\lVert\mathbf{v}\rVert_2\cos(\theta),
$$

cosine similarity equals

$$
\cos(\theta).
$$

It measures the angle between directions, not the ordinary Euclidean distance between endpoints.


### Detailed cosine-similarity example

Let

$$
\mathbf{u}=
\begin{bmatrix}
1\\2\\2
\end{bmatrix},
\qquad
\mathbf{v}=
\begin{bmatrix}
2\\1\\2
\end{bmatrix}.
$$

**Step 1: compute the dot product.**

$$
\mathbf{u}\cdot\mathbf{v}
=1(2)+2(1)+2(2)
=2+2+4
=8.
$$

**Step 2: compute the first L2 norm.**

$$
\lVert\mathbf{u}\rVert_2
=\sqrt{1^2+2^2+2^2}
=\sqrt{1+4+4}
=3.
$$

**Step 3: compute the second L2 norm.**

$$
\lVert\mathbf{v}\rVert_2
=\sqrt{2^2+1^2+2^2}
=\sqrt{4+1+4}
=3.
$$

**Step 4: divide.**

$$
\operatorname{cosim}(\mathbf{u},\mathbf{v})
=\frac{8}{(3)(3)}
=\frac{8}{9}
\approx 0.8889.
$$

The corresponding angle is

$$
\theta=\cos^{-1}\left(\frac{8}{9}\right)
\approx 27.27^{\circ}.
$$

The positive value close to `1` indicates strong directional alignment.


In [ ]:
u_cos = np.array([1.0, 2.0, 2.0])
v_cos = np.array([2.0, 1.0, 2.0])
dot_cos = np.dot(u_cos, v_cos)
norm_u = np.linalg.norm(u_cos)
norm_v = np.linalg.norm(v_cos)
cos_value = cosine_similarity(u_cos, v_cos)
angle_degrees = np.degrees(np.arccos(np.clip(cos_value, -1, 1)))

print("Dot product:", dot_cos)
print("Norm of u:", norm_u)
print("Norm of v:", norm_v)
print("Cosine similarity:", cos_value)
print("Angle in degrees:", angle_degrees)


## 3. Why cosine similarity stays between negative one and one

The Cauchy-Schwarz inequality states

$$
|\mathbf{u}\cdot\mathbf{v}|
\leq
\lVert\mathbf{u}\rVert_2\lVert\mathbf{v}\rVert_2.
$$

### Proof

Assume `v` is nonzero and define

$$
t=\frac{\mathbf{u}\cdot\mathbf{v}}
{\mathbf{v}\cdot\mathbf{v}}.
$$

Every squared norm is nonnegative, so

$$
0\leq\lVert\mathbf{u}-t\mathbf{v}\rVert_2^2.
$$

Use the dot product to expand:

$$
\lVert\mathbf{u}-t\mathbf{v}\rVert_2^2
=(\mathbf{u}-t\mathbf{v})\cdot(\mathbf{u}-t\mathbf{v}).
$$

$$
=\mathbf{u}\cdot\mathbf{u}
-2t(\mathbf{u}\cdot\mathbf{v})
+t^2(\mathbf{v}\cdot\mathbf{v}).
$$

Substitute the chosen value of `t`:

$$
0\leq
\mathbf{u}\cdot\mathbf{u}
-
\frac{(\mathbf{u}\cdot\mathbf{v})^2}
{\mathbf{v}\cdot\mathbf{v}}.
$$

Multiply by the positive number

$$
\mathbf{v}\cdot\mathbf{v}.
$$

Then

$$
(\mathbf{u}\cdot\mathbf{v})^2
\leq
(\mathbf{u}\cdot\mathbf{u})(\mathbf{v}\cdot\mathbf{v}).
$$

Because a self-dot product is a squared L2 norm,

$$
(\mathbf{u}\cdot\mathbf{v})^2
\leq
\lVert\mathbf{u}\rVert_2^2\lVert\mathbf{v}\rVert_2^2.
$$

Taking square roots yields the inequality. Dividing by the positive product of the two norms gives

$$
-1\leq
\operatorname{cosim}(\mathbf{u},\mathbf{v})
\leq 1.
$$


## 4. Interpreting cosine similarity

| Cosine similarity | Directional interpretation |
|---:|---|
| `1` | same direction |
| near `1` | strongly aligned |
| `0` | orthogonal |
| near `-1` | strongly opposed |
| `-1` | exactly opposite directions |
| undefined | at least one vector is zero |

> **Zero-vector warning**
>
> The formula divides by both L2 norms. The zero vector has L2 norm zero, so cosine similarity is undefined if either input vector is zero.


## 5. Scaling behavior

For a positive scalar `c`,

$$
\operatorname{cosim}(c\mathbf{u},\mathbf{v})
=
\frac{(c\mathbf{u})\cdot\mathbf{v}}
{\lVert c\mathbf{u}\rVert_2\lVert\mathbf{v}\rVert_2}.
$$

Use dot-product compatibility and norm homogeneity:

$$
=
\frac{c(\mathbf{u}\cdot\mathbf{v})}
{|c|\lVert\mathbf{u}\rVert_2\lVert\mathbf{v}\rVert_2}.
$$

When `c` is positive, `c` equals its absolute value, so the factors cancel:

$$
\operatorname{cosim}(c\mathbf{u},\mathbf{v})
=\operatorname{cosim}(\mathbf{u},\mathbf{v}).
$$

Positive scaling changes vector length but not direction. Negative scaling reverses direction and changes the sign of cosine similarity.


In [ ]:
base_u = np.array([1.0, 2.0])
base_v = np.array([2.0, 1.0])
for scale in [0.5, 2.0, 10.0, -2.0]:
    print(f"scale {scale:>4}: cosine = {cosine_similarity(scale*base_u, base_v): .4f}")


In [ ]:
# Visual and interactive cosine-similarity explorer.
def cosine_explorer(angle_degrees=45.0, scale=1.0):
    theta = np.radians(angle_degrees)
    u = np.array([1.0, 0.0])
    v = scale * np.array([np.cos(theta), np.sin(theta)])
    print(f"angle = {angle_degrees:.1f} degrees")
    print(f"length of v = {np.linalg.norm(v):.2f}")
    print(f"dot product = {np.dot(u, v):.4f}")
    print(f"cosine similarity = {cosine_similarity(u, v):.4f}")
    plot_two_vectors(u, v, title="Cosine depends on angle, not positive scale")

if WIDGETS_AVAILABLE:
    interact(
        cosine_explorer,
        angle_degrees=FloatSlider(value=45, min=0, max=180, step=5, description="angle"),
        scale=FloatSlider(value=1, min=0.25, max=5, step=0.25, description="scale"),
    )
else:
    cosine_explorer()


## 6. Direct AI application: embedding retrieval

### Background

An embedding model converts a query and each document chunk into vectors in the same feature space. A retrieval system then compares the query embedding with document embeddings and returns the highest-scoring chunks.

The conceptual workflow is:

1. split documents into chunks;
2. convert every chunk into an embedding;
3. convert the user's query into an embedding;
4. compute similarity scores;
5. rank the chunks; and
6. give the most relevant chunks to a generative model as context.

This is the retrieval step in **retrieval-augmented generation**, often abbreviated RAG.


### A transparent four-feature embedding space

Real embeddings usually have many learned coordinates whose individual meanings are not labeled. To make every calculation visible, this example uses four hand-designed coordinates in this order:

1. mathematics;
2. artificial intelligence;
3. music;
4. travel and hiking.

The query is represented by

$$
\mathbf{q}=
\begin{bmatrix}
0.8\\0.7\\0.1\\0
\end{bmatrix}.
$$

It emphasizes mathematics and AI, gives a small weight to music, and gives no weight to travel.


In [ ]:
feature_labels = ["mathematics", "AI", "music", "travel/hiking"]
query = np.array([0.8, 0.7, 0.1, 0.0])

document_names = [
    "Linear algebra for AI embeddings",
    "Large general mathematics archive",
    "R&B playlist recommendations",
    "Hiking route near Atlanta",
    "AI travel-planning assistant",
]

document_embeddings = np.array([
    [0.9, 0.8, 0.1, 0.0],
    [8.0, 0.2, 0.0, 0.0],
    [0.0, 0.1, 1.0, 0.0],
    [0.0, 0.0, 0.1, 1.0],
    [0.6, 0.6, 0.1, 0.7],
])

print("Query:", query)
print("Document matrix shape:", document_embeddings.shape)


### Detailed score for the first document

The first document embedding is

$$
\mathbf{d}_1=
\begin{bmatrix}
0.9\\0.8\\0.1\\0
\end{bmatrix}.
$$

Its dot product with the query is

$$
\mathbf{q}\cdot\mathbf{d}_1
=0.8(0.9)+0.7(0.8)+0.1(0.1)+0(0).
$$

$$
=0.72+0.56+0.01+0.
$$

$$
=1.29.
$$

The query norm is

$$
\lVert\mathbf{q}\rVert_2
=\sqrt{0.8^2+0.7^2+0.1^2+0^2}
=\sqrt{1.14}.
$$

The document norm is

$$
\lVert\mathbf{d}_1\rVert_2
=\sqrt{0.9^2+0.8^2+0.1^2+0^2}
=\sqrt{1.46}.
$$

Therefore,

$$
\operatorname{cosim}(\mathbf{q},\mathbf{d}_1)
=\frac{1.29}{\sqrt{1.14}\sqrt{1.46}}
\approx 0.9999.
$$

The two vectors point in nearly the same direction.


### Why raw dot-product ranking can be misleading

The second document is deliberately given a much larger magnitude:

$$
\mathbf{d}_2=
\begin{bmatrix}
8\\0.2\\0\\0
\end{bmatrix}.
$$

Its raw dot product is

$$
\mathbf{q}\cdot\mathbf{d}_2
=0.8(8)+0.7(0.2)
=6.4+0.14
=6.54.
$$

This is much larger than `1.29`, so raw dot product ranks the second document first. However, the second vector is dominated by only one coordinate and is less directionally aligned with the query. Cosine similarity corrects for its large magnitude.


In [ ]:
dot_scores = document_embeddings @ query
cosine_scores = np.array([cosine_similarity(query, row) for row in document_embeddings])

dot_order = np.argsort(-dot_scores)
cosine_order = np.argsort(-cosine_scores)

print("DOT-PRODUCT RANKING")
for rank, index in enumerate(dot_order, start=1):
    print(f"{rank}. {document_names[index]:38s} dot = {dot_scores[index]:.4f}")

print("\nCOSINE-SIMILARITY RANKING")
for rank, index in enumerate(cosine_order, start=1):
    print(f"{rank}. {document_names[index]:38s} cosine = {cosine_scores[index]:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
short_names = ["AI linear\nalgebra", "large math\narchive", "R&B\nplaylist", "hiking\nroute", "AI travel\nassistant"]

axes[0].barh(short_names, dot_scores, color="#f58518")
axes[0].set_title("Raw dot-product scores")
axes[0].set_xlabel("score")
axes[0].invert_yaxis()

axes[1].barh(short_names, cosine_scores, color="#4c78a8")
axes[1].set_title("Cosine-similarity scores")
axes[1].set_xlabel("score")
axes[1].set_xlim(0, 1.05)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 7. Normalization turns cosine similarity into a dot product

Normalize each nonzero vector to length one:

$$
\widehat{\mathbf{u}}
=\frac{\mathbf{u}}{\lVert\mathbf{u}\rVert_2},
\qquad
\widehat{\mathbf{v}}
=\frac{\mathbf{v}}{\lVert\mathbf{v}\rVert_2}.
$$

Then

$$
\widehat{\mathbf{u}}\cdot\widehat{\mathbf{v}}
=
\frac{\mathbf{u}\cdot\mathbf{v}}
{\lVert\mathbf{u}\rVert_2\lVert\mathbf{v}\rVert_2}
=\operatorname{cosim}(\mathbf{u},\mathbf{v}).
$$

This is why embedding systems often normalize vectors once and then use fast dot products during retrieval.


In [ ]:
query_unit = query / np.linalg.norm(query)
document_norms = np.linalg.norm(document_embeddings, axis=1, keepdims=True)
documents_unit = document_embeddings / document_norms
normalized_dot_scores = documents_unit @ query_unit

print("Cosine scores:       ", np.round(cosine_scores, 6))
print("Normalized dot scores:", np.round(normalized_dot_scores, 6))
print("Maximum absolute difference:", np.max(np.abs(cosine_scores - normalized_dot_scores)))


In [ ]:
# Pairwise similarity heatmap for the query and all documents.
all_names = ["query", *short_names]
all_vectors = np.vstack([query, document_embeddings])
all_unit = all_vectors / np.linalg.norm(all_vectors, axis=1, keepdims=True)
similarity_matrix = all_unit @ all_unit.T

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(similarity_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(all_names)), all_names, rotation=40, ha="right")
ax.set_yticks(range(len(all_names)), all_names)
ax.set_title("Pairwise cosine similarities")
for i in range(len(all_names)):
    for j in range(len(all_names)):
        ax.text(j, i, f"{similarity_matrix[i,j]:.2f}", ha="center", va="center",
                color="white" if abs(similarity_matrix[i,j]) > 0.55 else "black")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()


In [ ]:
# Interactive retrieval: change the conceptual query coordinates.
def retrieve_with_query(math_weight=0.8, ai_weight=0.7, music_weight=0.1, travel_weight=0.0):
    q = np.array([math_weight, ai_weight, music_weight, travel_weight], dtype=float)
    if np.isclose(np.linalg.norm(q), 0):
        print("The query vector is zero, so cosine similarity is undefined.")
        return
    scores = np.array([cosine_similarity(q, row) for row in document_embeddings])
    order = np.argsort(-scores)
    print("Query vector:", q)
    print("Top retrieval results:")
    for rank, index in enumerate(order[:3], start=1):
        print(f"{rank}. {document_names[index]:38s} score = {scores[index]:.4f}")

if WIDGETS_AVAILABLE:
    interact(
        retrieve_with_query,
        math_weight=FloatSlider(value=0.8, min=0, max=1, step=0.1, description="math"),
        ai_weight=FloatSlider(value=0.7, min=0, max=1, step=0.1, description="AI"),
        music_weight=FloatSlider(value=0.1, min=0, max=1, step=0.1, description="music"),
        travel_weight=FloatSlider(value=0.0, min=0, max=1, step=0.1, description="travel"),
    )
else:
    retrieve_with_query()


## 8. Limitations of cosine similarity

Cosine similarity is useful, but it is not a guarantee of semantic truth.

1. It ignores overall magnitude by design.
2. Its quality depends on the embedding model and training data.
3. High similarity means the vectors are aligned in the chosen representation; it does not prove factual correctness.
4. Different embedding models may use different spaces, so their vectors should not be mixed without justification.
5. A retrieval system must still choose the number of chunks, handle ties, and evaluate whether the returned context is useful.


### Classroom check

Let

$$
\mathbf{u}=
\begin{bmatrix}
1\\1
\end{bmatrix},
\qquad
\mathbf{v}=
\begin{bmatrix}
5\\5
\end{bmatrix},
\qquad
\mathbf{w}=
\begin{bmatrix}
1\\-1
\end{bmatrix}.
$$

1. Compute the cosine similarity between `u` and `v`.
2. Compute the cosine similarity between `u` and `w`.
3. Explain why the first answer is not affected by the different lengths.


In [ ]:
reveal_markdown("Reveal instructor solution", r'''
### 1. Cosine between u and v

$$
\mathbf{u}\cdot\mathbf{v}=1(5)+1(5)=10.
$$

$$
\lVert\mathbf{u}\rVert_2=\sqrt{2},
\qquad
\lVert\mathbf{v}\rVert_2=\sqrt{50}=5\sqrt{2}.
$$

Therefore,

$$
\operatorname{cosim}(\mathbf{u},\mathbf{v})
=\frac{10}{(\sqrt{2})(5\sqrt{2})}
=\frac{10}{10}
=1.
$$

### 2. Cosine between u and w

$$
\mathbf{u}\cdot\mathbf{w}=1(1)+1(-1)=0.
$$

Both vectors are nonzero, so

$$
\operatorname{cosim}(\mathbf{u},\mathbf{w})=0.
$$

### 3. Length explanation

The vector `v` is exactly five times `u`. Positive scaling changes length but not direction, and the scale factor cancels between the numerator and denominator of cosine similarity.
''')


> **Common mistakes in Section 4**
>
> - Dividing the dot product by only one norm.
> - Using L1 norms in the standard cosine-similarity formula.
> - Computing cosine similarity when one vector is zero.
> - Assuming a large raw dot product always means the closest direction.
> - Interpreting a cosine score as a probability.
> - Comparing embeddings produced by unrelated models as though their coordinates shared one space.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Cosine similarity | Normalized dot product measuring directional alignment |
| Unit vector | Vector with L2 norm one |
| Normalization | Dividing a nonzero vector by its norm |
| Query embedding | Vector representation of a search request |
| Document embedding | Vector representation of a document or chunk |
| Retrieval | Ranking and selecting relevant stored items |
| RAG | Retrieval-augmented generation |


## Section 4 worksheet

1. Compute the cosine similarity between

$$
\mathbf{a}=
\begin{bmatrix}
3\\4
\end{bmatrix}
$$

and

$$
\mathbf{b}=
\begin{bmatrix}
4\\3
\end{bmatrix}.
$$

2. Without recomputing from scratch, determine the cosine similarity between `10a` and `b`.
3. Explain why cosine similarity with the zero vector is undefined.
4. A query has cosine scores `0.82`, `0.31`, and `0.76` with three chunks. Rank the chunks from most to least similar.


In [ ]:
reveal_markdown("Open Section 4 solutions", r'''
### 1. Cosine calculation

$$
\mathbf{a}\cdot\mathbf{b}=3(4)+4(3)=12+12=24.
$$

$$
\lVert\mathbf{a}\rVert_2=\sqrt{3^2+4^2}=5,
$$

$$
\lVert\mathbf{b}\rVert_2=\sqrt{4^2+3^2}=5.
$$

Therefore,

$$
\operatorname{cosim}(\mathbf{a},\mathbf{b})
=\frac{24}{(5)(5)}
=\frac{24}{25}
=0.96.
$$

### 2. Positive scaling

Multiplying `a` by positive `10` does not change its direction, so the cosine similarity remains `0.96`.

### 3. Zero vector

The L2 norm of the zero vector is zero. The cosine formula would divide by zero, so the value is undefined.

### 4. Ranking

From highest to lowest score, the order is: first chunk (`0.82`), third chunk (`0.76`), second chunk (`0.31`).
''')


---

# Integrated Instructor Lab: Build a Miniature Retrieval System

This lab combines arrays, dot products, norms, distances, normalization, and cosine ranking. It uses transparent handcrafted embeddings so every number can be audited.

## Scenario

A small advising assistant stores four course-resource chunks. Coordinates appear in this order:

1. linear algebra;
2. calculus;
3. probability;
4. graph theory.

The query is “Which resource explains vector similarity for AI?”

Use the query embedding

$$
\mathbf{q}=
\begin{bmatrix}
0.9\\0.1\\0.1\\0.2
\end{bmatrix}.
$$


In [ ]:
lab_chunk_names = [
    "Vectors and cosine similarity",
    "Chain rule and backpropagation",
    "Bayes theorem for classification",
    "Graph traversal for pathfinding",
]

lab_embeddings = np.array([
    [0.95, 0.10, 0.05, 0.10],
    [0.10, 0.95, 0.05, 0.05],
    [0.05, 0.10, 0.95, 0.05],
    [0.10, 0.05, 0.10, 0.95],
])

lab_query = np.array([0.9, 0.1, 0.1, 0.2])

print("Embedding matrix shape:", lab_embeddings.shape)
print("Query shape:", lab_query.shape)
print("Each row and the query have four coordinates, so comparisons are compatible.")


## Lab tasks

1. Compute every query-document dot product.
2. Compute the L2 norm of the query and each document.
3. Normalize the query and documents.
4. Compute cosine similarities two ways:
   - directly from the cosine formula;
   - as dot products of normalized vectors.
5. Rank the resources.
6. Compute L2 distances between the normalized query and normalized documents.
7. Explain why the highest cosine similarity also gives the smallest normalized L2 distance in this example.


In [ ]:
# Complete instructor solution, displayed computationally and step by step.
lab_dot_scores = lab_embeddings @ lab_query
lab_query_norm = np.linalg.norm(lab_query)
lab_document_norms = np.linalg.norm(lab_embeddings, axis=1)
lab_cosines_direct = lab_dot_scores / (lab_document_norms * lab_query_norm)

lab_query_unit = lab_query / lab_query_norm
lab_documents_unit = lab_embeddings / lab_document_norms[:, None]
lab_cosines_normalized = lab_documents_unit @ lab_query_unit
lab_normalized_distances = np.linalg.norm(lab_documents_unit - lab_query_unit, axis=1)
lab_order = np.argsort(-lab_cosines_direct)

print("QUERY NORM")
print(f"sqrt(0.9² + 0.1² + 0.1² + 0.2²) = {lab_query_norm:.6f}\n")

for i, name in enumerate(lab_chunk_names):
    products = lab_embeddings[i] * lab_query
    print(f"CHUNK {i+1}: {name}")
    print("  coordinate products:", products)
    print(f"  dot product: {products.sum():.6f}")
    print(f"  document norm: {lab_document_norms[i]:.6f}")
    print(f"  cosine similarity: {lab_cosines_direct[i]:.6f}")
    print(f"  normalized L2 distance: {lab_normalized_distances[i]:.6f}\n")

print("RANKING")
for rank, index in enumerate(lab_order, start=1):
    print(f"{rank}. {lab_chunk_names[index]:38s} cosine = {lab_cosines_direct[index]:.6f}")

print("\nAgreement check between the two cosine methods:")
print(np.allclose(lab_cosines_direct, lab_cosines_normalized))


### Why cosine and normalized L2 distance give opposite rankings

For unit vectors,

$$
\lVert\widehat{\mathbf{u}}-\widehat{\mathbf{v}}\rVert_2^2
=(\widehat{\mathbf{u}}-\widehat{\mathbf{v}})
\cdot
(\widehat{\mathbf{u}}-\widehat{\mathbf{v}}).
$$

Expand the dot product:

$$
=\widehat{\mathbf{u}}\cdot\widehat{\mathbf{u}}
-2\widehat{\mathbf{u}}\cdot\widehat{\mathbf{v}}
+\widehat{\mathbf{v}}\cdot\widehat{\mathbf{v}}.
$$

Both self-dot products equal one because the vectors are normalized:

$$
=1-2\widehat{\mathbf{u}}\cdot\widehat{\mathbf{v}}+1.
$$

Therefore,

$$
\lVert\widehat{\mathbf{u}}-\widehat{\mathbf{v}}\rVert_2^2
=2-2\operatorname{cosim}(\mathbf{u},\mathbf{v}).
$$

As cosine similarity increases, normalized squared L2 distance decreases. Thus the two rankings are equivalent when every vector is normalized.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x_positions = np.arange(len(lab_chunk_names))
width = 0.38
ax.bar(x_positions - width/2, lab_cosines_direct, width, label="cosine similarity", color="#4c78a8")
ax.bar(x_positions + width/2, lab_normalized_distances, width, label="normalized L2 distance", color="#e45756")
ax.set_xticks(x_positions, ["vectors", "chain rule", "Bayes", "graphs"])
ax.set_ylabel("score or distance")
ax.set_title("Best retrieval: highest cosine and smallest normalized distance")
ax.legend()
plt.show()


> **Instructor discussion prompts**
>
> 1. What would fail if the query had five coordinates while the documents had four?
> 2. Why is the embedding matrix a matrix rather than a tensor of order three?
> 3. Which calculation uses a scalar output but two vector inputs?
> 4. What information is lost when vectors are normalized?
> 5. Why should these handcrafted coordinates not be mistaken for how a production embedding model exposes meaning?


---

# Module 1 Wrap-Up

## Mathematical storyline

1. **Representation:** Scalars, vectors, matrices, and tensors organize numerical data.
2. **Interaction:** A dot product combines matching vector coordinates into one scalar.
3. **Geometry:** Dot products reveal angles, orthogonality, and projections.
4. **Magnitude:** L1 and L2 norms measure vector size in different ways.
5. **Separation:** Distances measure the norm of a vector difference.
6. **Model control:** Norm-based regularization discourages overly large weights.
7. **Similarity:** Cosine similarity removes positive scale and compares direction.
8. **Retrieval:** Embedding systems use these ideas to rank relevant information.


## Formula reference

### Dot product

$$
\mathbf{u}\cdot\mathbf{v}
=\sum_{i=1}^{n}u_iv_i.
$$

### Orthogonality

$$
\mathbf{u}\perp\mathbf{v}
\quad\Longleftrightarrow\quad
\mathbf{u}\cdot\mathbf{v}=0.
$$

### Projection

$$
\operatorname{proj}_{\mathbf{u}}(\mathbf{v})
=\frac{\mathbf{v}\cdot\mathbf{u}}
{\mathbf{u}\cdot\mathbf{u}}\mathbf{u}.
$$

### L1 norm

$$
\lVert\mathbf{x}\rVert_1
=\sum_{i=1}^{n}|x_i|.
$$

### L2 norm

$$
\lVert\mathbf{x}\rVert_2
=\sqrt{\sum_{i=1}^{n}x_i^2}.
$$

### Distance

$$
d_p(\mathbf{x},\mathbf{y})
=\lVert\mathbf{x}-\mathbf{y}\rVert_p.
$$

### Cosine similarity

$$
\operatorname{cosim}(\mathbf{u},\mathbf{v})
=\frac{\mathbf{u}\cdot\mathbf{v}}
{\lVert\mathbf{u}\rVert_2\lVert\mathbf{v}\rVert_2}.
$$


## Reusable NumPy reference

| Mathematical task | NumPy expression |
|---|---|
| Create a vector | `np.array([1, 2, 3])` |
| Inspect shape | `x.shape` |
| Dot product | `np.dot(u, v)` or `u @ v` |
| L1 norm | `np.linalg.norm(x, ord=1)` |
| L2 norm | `np.linalg.norm(x, ord=2)` |
| L2 distance | `np.linalg.norm(x - y, ord=2)` |
| Normalize | `x / np.linalg.norm(x)` |
| Batch dot products | `matrix @ vector` |

Code should follow, not replace, the mathematical reasoning. Check dimensions and zero denominators before calculating.


In [ ]:
# Compact reusable reference functions.
def l1_norm(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(np.abs(x)))


def l2_norm(x):
    x = np.asarray(x, dtype=float)
    return float(np.sqrt(np.sum(x**2)))


def l2_distance(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.shape != y.shape:
        raise ValueError("Vectors must have the same shape.")
    return l2_norm(x - y)


def normalize(x):
    x = np.asarray(x, dtype=float)
    length = l2_norm(x)
    if np.isclose(length, 0.0):
        raise ValueError("The zero vector cannot be normalized.")
    return x / length


print("Reusable reference functions are ready.")


## Cumulative practice

Let

$$
\mathbf{x}=
\begin{bmatrix}
2\\-1\\2
\end{bmatrix},
\qquad
\mathbf{y}=
\begin{bmatrix}
1\\2\\2
\end{bmatrix}.
$$

1. State the dimension and NumPy shape of each vector.
2. Compute the dot product.
3. Determine whether the vectors are orthogonal.
4. Compute both L1 norms.
5. Compute both L2 norms.
6. Compute the L1 distance.
7. Compute the L2 distance.
8. Compute cosine similarity.
9. Explain whether the vectors are more aligned, orthogonal, or opposed.


In [ ]:
reveal_markdown("Open cumulative solution", r'''
### 1. Dimension and shape

Each vector has dimension `3`. In NumPy, each one-dimensional array has shape `(3,)`.

### 2. Dot product

$$
\mathbf{x}\cdot\mathbf{y}
=2(1)+(-1)(2)+2(2)
=2-2+4
=4.
$$

### 3. Orthogonality

The dot product is not zero, so the vectors are not orthogonal.

### 4. L1 norms

$$
\lVert\mathbf{x}\rVert_1
=|2|+|-1|+|2|
=5,
$$

$$
\lVert\mathbf{y}\rVert_1
=|1|+|2|+|2|
=5.
$$

### 5. L2 norms

$$
\lVert\mathbf{x}\rVert_2
=\sqrt{2^2+(-1)^2+2^2}
=\sqrt{4+1+4}
=3,
$$

$$
\lVert\mathbf{y}\rVert_2
=\sqrt{1^2+2^2+2^2}
=\sqrt{1+4+4}
=3.
$$

### 6. L1 distance

$$
\mathbf{x}-\mathbf{y}
=
\begin{bmatrix}
1\\-3\\0
\end{bmatrix}.
$$

Therefore,

$$
d_1=|1|+|-3|+|0|=4.
$$

### 7. L2 distance

$$
d_2=\sqrt{1^2+(-3)^2+0^2}=\sqrt{10}.
$$

### 8. Cosine similarity

$$
\operatorname{cosim}(\mathbf{x},\mathbf{y})
=\frac{4}{(3)(3)}
=\frac{4}{9}
\approx 0.4444.
$$

### 9. Interpretation

The cosine similarity is positive, so the vectors are more aligned than opposed. It is not close to one, so the alignment is moderate rather than strong.
''')


## Exit ticket

Ask students to answer without code.

1. In one sentence, distinguish a matrix from a vector.
2. What does a zero dot product mean?
3. What is the main computational difference between the L1 and L2 norms?
4. Why does cosine similarity require nonzero vectors?
5. Name one direct AI use of a concept from this module.


In [ ]:
reveal_markdown("Reveal suggested exit-ticket answers", r'''
1. A vector is a one-dimensional ordered list, while a matrix is a two-dimensional rectangular array.
2. A zero dot product means the vectors are orthogonal, provided the usual same-dimensional dot product is defined.
3. L1 adds absolute values; L2 squares entries, adds them, and takes a square root.
4. Cosine similarity divides by both vector norms, and the zero vector has norm zero.
5. Examples include embedding retrieval, RAG, linear-neuron scores, vector-distance comparisons, and L1 or L2 regularization.
''')
